# Building Physics

## Example: Describe a building envelope and a thermal zone

In this example, we will add a building envelope and a thermal zone to a building manually. In practice, this could be cumbersome. There exist tools to automate that process and read in data from file.

#### Set things up

Required imports:

In [15]:
from odeon.model import (
    Project,
    Branch,
    Building,
    Household,
    BuildingThermalZone,
    AdjacentEnvironment,
    MassDistribution,
    Transparency,
    NominalGeometry,
    Wall,
    WallPhysics,
    Roof,
    RoofPhysics,
    Window,
    WindowPhysics,
    WindowType,
    Floor,
    FloorPhysics,
    Door,
    DoorPhysics,
    CardinalOrientation,
    OpaqueElementPhysics,
)

Create a project `my_project` and a branch `my_branch`:

In [2]:
my_project = Project()
my_branch = Branch(year=2023)
my_project.main_branch = my_branch

Initialize an instance of a Building `my_building` and a Household `my_household`:

In [3]:
my_building = Building()
my_building.usable_area = 280
my_household = Household()
my_household.net_floor_area = 240

my_building.add_building_units(my_household)

my_branch.add_objects(my_building)

#### Create a thermal zone

The object `BuildingThermalZone` is used to describe general attributes of a thermal zone and its operation. Until now, a Building can have only one thermal zone:

In [4]:
btz = BuildingThermalZone()
btz.heated_area = my_household.net_floor_area
btz.air_exchange_rate_use = 0.2
btz.internal_heat_capacity_j_per_k = my_household.net_floor_area * 10000
btz.heattranscoef_thermal_bridges_w_per_sqm_k = 0.1
btz.heated_volume = my_household.net_floor_area * 3

my_building.building_thermal_zone = btz

#### Create a building element

Let's define a building element from scratch, for example a wall:

In [5]:
wall_1 = Wall()

wall_1.adjacent_environment = AdjacentEnvironment.AIR

The wall needs a geometry description. This could be given by exact coordinates, or in a more abstract way by a `NominalGeometry` which is sufficient for our use case:

In [6]:
wall_1.element_geometry = NominalGeometry(
    azimuth=90,
    tilt=90,
    dimensions_nominal=(20, 9)
)

Finally, we can create a `WallPhysics` taking in a bunch of attributes required for thermal simulations:

In [7]:
wall_1.element_physics = WallPhysics()

wall_1.element_physics.u_value_w_per_sqm_k = 0.3
wall_1.element_physics.specific_heat_capacity_j_per_sqm_k = 110000
wall_1.element_physics.mass_distribution = MassDistribution.MASS_CONCENTRATED_INSIDE
wall_1.element_physics.transparency = Transparency.OPAQUE
wall_1.element_physics.view_factor = 0.5
wall_1.element_physics.shading_factor = 0.6

#### Create more building elements

Let's create additional building elements:

- a roof,
- a window,
- a floor,
- and a door.

With only one wall and one window, our building won't be too cosy; for our example, however, it's sufficient.

In [8]:
roof_1 = Roof()

roof_1.adjacent_environment = AdjacentEnvironment.AIR
roof_1.element_geometry = NominalGeometry(
    azimuth=90,
    tilt=25,
    dimensions_nominal=(20, 9)
)

roof_1.element_physics = RoofPhysics()

roof_1.element_physics.u_value_w_per_sqm_k = 0.3
roof_1.element_physics.specific_heat_capacity_j_per_sqm_k = 50000
roof_1.element_physics.mass_distribution = MassDistribution.MASS_CONCENTRATED_INSIDE
roof_1.element_physics.transparency = Transparency.OPAQUE
roof_1.element_physics.view_factor = 0.5
roof_1.element_physics.shading_factor = 0.6

In [9]:
window_1 = Window()

window_1.adjacent_environment = AdjacentEnvironment.AIR
window_1.element_geometry = NominalGeometry(
    azimuth=90,
    tilt=25,
    dimensions_nominal=(10, 3)
)

window_1.element_physics = WindowPhysics()

window_1.element_physics.u_value_w_per_sqm_k = 1.5
window_1.element_physics.transparency = Transparency.TRANSPARENT
window_1.element_physics.view_factor = 0
window_1.element_physics.shading_factor = 0.2
window_1.element_physics.frame_portion = 0.2
window_1.element_physics.window_type = WindowType.DOUBLE_THERMAL_INSULATION_GLAZING
window_1.element_physics.total_solar_energy_transmittance = 0.5

In [10]:
floor = Floor()

floor.adjacent_environment = AdjacentEnvironment.GROUND
floor.element_geometry = NominalGeometry(
    azimuth=90,
    tilt=0,
    dimensions_nominal=(10, 10)
)

floor.element_physics = FloorPhysics()

floor.element_physics.u_value_w_per_sqm_k = 0.5
floor.element_physics.specific_heat_capacity_j_per_sqm_k = 110000
floor.element_physics.mass_distribution = MassDistribution.MASS_CONCENTRATED_INSIDE
floor.element_physics.transparency = Transparency.OPAQUE
floor.element_physics.view_factor = 0
floor.element_physics.shading_factor = 0

In [11]:
door = Door()

door.adjacent_environment = AdjacentEnvironment.AIR
door.element_geometry = NominalGeometry(
    azimuth = 90,
    tilt=90,
    dimensions_nominal=(1, 2)
)

door.element_physics = DoorPhysics()

door.element_physics.u_value_w_per_sqm_k = 0.5
door.element_physics.transparency = Transparency.TRANSPARENT
door.element_physics.view_factor = 0.5
door.element_physics.shading_factor = 0.2

Ultimately, we can add the created building elements to our building. These will form our building's envelope.

In [12]:
my_building.add_building_elements(
    [
        wall_1,
        window_1,
        roof_1,
        floor,
        door,
    ]
)

Ode's plotting tool *Incide* comes with a method to print an envelope table:

In [13]:
import pandas as pd

records_envelope = []
for be in my_building.building_elements:
    record = {
        "Element id": be.id,
        "Element type": be.__class__.__name__,
        "Material": be.element_physics.material,
        "Element area (m²)": be.element_geometry.area,
        "Tilt (°)": be.element_geometry.tilt,
        "Azimuth (°)": be.element_geometry.azimuth,
        "Azimuth, cardinal": CardinalOrientation.by_closest_degrees(be.element_geometry.azimuth).letter,
        "Heat transfer coefficient (W/(m²K))": be.element_physics.u_value_w_per_sqm_k,
    }
    if isinstance(be.element_physics, OpaqueElementPhysics):
        record |= {
            "Specific heat capacity (J/(m²K))": be.element_physics.specific_heat_capacity_j_per_sqm_k,
            "Insulation thickness (m)": be.element_physics.insulation_thickness,
        }
    records_envelope.append(record)

pd.DataFrame.from_records(records_envelope)

,Element id,Element type,Material,Element area (m²),Tilt (°),Azimuth (°),"Azimuth, cardinal",Heat transfer coefficient (W/(m²K)),Specific heat capacity (J/(m²K)),Insulation thickness (m)
0,17,Wall,None,180,90,90,E,0.3,110000.0,NaN
1,25,Roof,None,180,25,90,E,0.3,50000.0,NaN
2,33,Window,None,30,25,90,E,1.5,NaN,NaN
3,41,Floor,None,100,0,90,E,0.5,110000.0,NaN
4,51,Door,None,2,90,90,E,0.5,NaN,NaN


Together with the `BuildingThermalZone`, all these elements are stored as children of the Building:

In [14]:
from odeon.processing.prints import print_object_tree, print_project_stats

print("Project Overview:\n")
print_project_stats(my_project)

print("\nBranch overview:\n")
print_object_tree(my_project.main_branch)

Project Overview:

└─<Project>: None
  │ 
  └─<Project>.branch[0](main): (no name)
      - description = []
      - validity year = 2023
      - no. of base objects = 1
      - no. of objects = 16
      

Branch overview:

└─Branch(id=0, n_objects=1, year=2023)
  └─Building(id=2)
    ├─FootprintNominalBuildingGeometry(id=1)
    ├─EnergySystem(id=3)
    ├─Household(id=4)
    │ └─EnergySystem(id=5)
    ├─BuildingThermalZone(id=11)
    ├─Wall(id=17)
    │ └─WallPhysics(id=21)
    ├─Roof(id=25)
    │ └─RoofPhysics(id=29)
    ├─Window(id=33)
    │ └─WindowPhysics(id=37)
    ├─Floor(id=41)
    │ └─FloorPhysics(id=46)
    └─Door(id=51)
      └─DoorPhysics(id=55)
